# Download the paired AbNGS database

the csv files are downloaded from https://naturalantibody.com/paired-ab-ngs/ (.airr version dataset)
<p>download the files to a directory 
<p>the directory path is inserted in the following script where "INSERT_PATH_TO_DATABASE" is written

In [ ]:
from pathlib import Path
import polars as pl


PROJECT_DIR = Path("INSERT_PATH_TO_DATABASE")
RESULTS = Path("./")
airr = pl.scan_csv(
    str(PROJECT_DIR / "bioproject=*" / "*.csv"),
    separator="\t",
    infer_schema_length=1000,
    low_memory=True,
    schema_overrides={"sra": pl.String},
)


# write fastas

In [ ]:


# all heavy chain sequences from the database
sequences = (
    airr
    .filter(pl.col('v_call').str.contains('HV'))
    .select("sequence_id", "sra", "sequence_alignment")
    .collect(engine="streaming")
)
output_path = RESULTS / f"sequences_heavy.fasta"
print(f"Writing {len(sequences)} ")

with output_path.open("w") as fasta:
    for row in sequences.iter_rows(named=True):
        sequence = row["sequence_alignment"]
        if not row["sequence_id"] or not row["sra"] or not sequence:
            continue
        sequence_id = f"{row['sequence_id']}_{row['sra']}"
        fasta.write(f">{sequence_id}\n{sequence}\n")

# all lambda light chain sequences from the database
sequences = (
    airr
    .filter((pl.col('v_call').str.contains('LV')))
    .select("sequence_id", "sra", "sequence_alignment")
    .collect(engine="streaming")
)
output_path = RESULTS / f"sequences_lambda.fasta"
print(f"Writing {len(sequences)} ")

with output_path.open("w") as fasta:
    for row in sequences.iter_rows(named=True):
        sequence = row["sequence_alignment"]
        if not row["sequence_id"] or not row["sra"] or not sequence:
            continue
        sequence_id = f"{row['sequence_id']}_{row['sra']}"
        fasta.write(f">{sequence_id}\n{sequence}\n")

# all kappa light chain sequences from the database
sequences = (
    airr
    .filter((pl.col('v_call').str.contains('KV')))
    .select("sequence_id", "sra", "sequence_alignment")
    .collect(engine="streaming")
)
output_path = RESULTS / f"sequences_kappa.fasta"
print(f"Writing {len(sequences)} ")

with output_path.open("w") as fasta:
    for row in sequences.iter_rows(named=True):
        sequence = row["sequence_alignment"]
        if not row["sequence_id"] or not row["sra"] or not sequence:
            continue
        sequence_id = f"{row['sequence_id']}_{row['sra']}"
        fasta.write(f">{sequence_id}\n{sequence}\n")